In [13]:
# Importing

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

In [14]:
# Loading the Titanic data and Feature Engineering

df = pd.read_csv('train.csv')
df.drop('Cabin', axis=1, inplace=True)

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
df['Title'] = df['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
)

df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

df.drop(['Name','Ticket','PassengerId'], axis=1, inplace=True)

X = df.drop('Survived', axis=1)
y = df['Survived']

print(X.dtypes)
print(X.head())

Pclass          int64
Sex               str
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked          str
FamilySize      int64
IsAlone         int64
Title             str
dtype: object
   Pclass     Sex   Age  SibSp  Parch     Fare Embarked  FamilySize  IsAlone  \
0       3    male  22.0      1      0   7.2500        S           2        0   
1       1  female  38.0      1      0  71.2833        C           2        0   
2       3  female  26.0      0      0   7.9250        S           1        1   
3       1  female  35.0      1      0  53.1000        S           2        0   
4       3    male  35.0      0      0   8.0500        S           1        1   

  Title  
0    Mr  
1   Mrs  
2  Miss  
3   Mrs  
4    Mr  


In [15]:
# Defining numeric and categorical columns

num_cols = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch', 'IsAlone']
cat_cols = ['Sex', 'Embarked', 'Title', 'Pclass']

In [16]:
# Building Numberic Pipline

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [17]:
# Building Categorical Pipline

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [18]:
# Combining with ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [24]:
# Full Pipeline with model

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

In [29]:
# Train ans evaluating

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42
                                                   ,stratify=y)
full_pipeline.fit(X_train,y_train)

prediction = full_pipeline.predict(X_test)

print(f'Validation Accuracy: {accuracy_score(y_test,prediction):.4f}')

scores = cross_val_score(full_pipeline, X, y, cv = 5, scoring='accuracy')

print(f'CV Accuracy: {scores.mean():.4f} +- {scores.std():.4f}')

Validation Accuracy: 0.8324
CV Accuracy: 0.8305 +- 0.0188


In [31]:
# Save pipeline with joblib

import joblib
joblib.dump(full_pipeline, 'titanic_pipeline.pkl')
print('Pipeline Saved!')

# Loading and testing

loaded = joblib.load('titanic_pipeline.pkl')
print(f'Loaded Pipeline accuracy: {accuracy_score(y_test, loaded.predict(X_test)):.4f}')

Pipeline Saved!
Loaded Pipeline accuracy: 0.8324
